# 06 Binoculars

Binoculars is zero-shot, so there is nothing to train. `scripts/run_binoculars.py`
scored every calibration and test text once on Viking, and this notebook turns
those scores into results.

That makes it much shorter than notebook 05. There are no training sets and no
seeds, so there is one threshold and one row per family rather than forty runs.
Binoculars is also unaffected by generator shift by construction, since no family
can be withheld from training it never did, which is what makes it the reference
line the trained detectors are measured against (section 3.5).

## 1. Check the metrics signatures

The cells below call four functions from `logos.evaluation.metrics`. Print their
signatures first, so a mismatch shows up here rather than halfway down the
notebook.

In [1]:
import inspect

from logos.evaluation import metrics

for name in ["calibrate_threshold", "tpr_at_threshold", "fpr_at_threshold", "auroc"]:
    print(name, inspect.signature(getattr(metrics, name)))

calibrate_threshold (scores, labels, target_fpr=0.01)
tpr_at_threshold (scores, labels, tau)
fpr_at_threshold (scores, labels, tau)
auroc (scores, labels)


## 2. Load the corpus and the scores

The scores file has one row per scored text, with `text_id` to join on, `part` to
say whether the text came from the calibration or the test split, and `score`.

The scores were sign-flipped when they were saved, so a higher score means more
likely machine-generated, matching every other detector in the study. A raw
Binoculars score runs the other way.

In [2]:
import numpy as np
import pandas as pd

from logos.paths import DATA_DIR
from logos.evaluation.folds import calibration_set, test_set
from logos.evaluation.metrics import (
    calibrate_threshold, tpr_at_threshold, fpr_at_threshold, auroc
)

corpus = pd.read_parquet(DATA_DIR / "mage" / "corpus.parquet")
scores = pd.read_parquet(DATA_DIR / "mage" / "scores" / "binoculars" / "all.parquet")

print(f"corpus {len(corpus)} rows, scores {len(scores)} rows")
scores.head()

corpus 392385 rows, scores 80882 rows


,text_id,part,score
0,287357,calibration,-1.022346
1,287358,calibration,-0.995238
2,287359,calibration,-0.978836
3,287360,calibration,-0.947619
4,287361,calibration,-1.000000


## 3. Coverage and direction checks

Two things to confirm before reading anything into the numbers.

First, every calibration and test text should pick up exactly one score. If the
counts do not match, the join has either lost texts or duplicated them.

Second, machine texts should score higher than human texts on average. If they
do not, the sign flip went astray and everything below is meaningless.

In [3]:
calibration = calibration_set(corpus).merge(
    scores[scores["part"] == "calibration"], on="text_id"
)
test = test_set(corpus).merge(scores[scores["part"] == "test"], on="text_id")

print(f"calibration {len(calibration)} of {len(calibration_set(corpus))}")
print(f"test        {len(test)} of {len(test_set(corpus))}")

print(f"\nmean score, human test texts   {test[test['label'] == 0]['score'].mean():.4f}")
print(f"mean score, machine test texts {test[test['label'] == 1]['score'].mean():.4f}")

calibration 28407 of 28407
test        52475 of 52475

mean score, human test texts   -0.9708
mean score, machine test texts -0.9381


## 4. Calibrate the threshold

One threshold for everything. The trained detectors need a threshold per training
set, because each training set produces a different model and therefore a
different spread of scores. Binoculars produces the same scores whatever is
withheld, so it has a single threshold.

The calibration texts are the human texts from MAGE's validation split, which are
available at deployment in a way text from an unseen generator is not.

Expect something around -0.85 to -0.95. The authors of Binoculars published a
threshold of 0.8536 for their own data at a much stricter error rate, and negating
it for the sign flip gives roughly this region. A value far outside it points to a
problem in the scoring rather than in the calibration.

In [4]:
threshold = calibrate_threshold(
    calibration["score"].to_numpy(), calibration["label"].to_numpy()
)
print(f"tau* = {threshold:.6f}")

tau* = -0.734043


## 5. Per-family results

Each family is tested on the shared human test texts plus that family's machine
texts, exactly as in notebook 05. TPR is the share of that family's machine texts
caught at the frozen threshold, and AUROC is computed on the same set as a
secondary measure for RQ1.

`n_machine` is kept because the size of a family's test set sets how precise its
TPR can be. GLM-130B is the smallest at 919 texts.

In [5]:
rows = []
for family in sorted(corpus[corpus["label"] == 1]["family"].dropna().unique()):
    subset = test_set(corpus, target=family).merge(
        scores[scores["part"] == "test"], on="text_id"
    )
    family_scores = subset["score"].to_numpy()
    family_labels = subset["label"].to_numpy()

    rows.append({
        "detector": "binoculars",
        "family": family,
        "condition": "Zero-shot",
        "tpr": tpr_at_threshold(family_scores, family_labels, threshold),
        "auroc": auroc(family_scores, family_labels),
        "test_fpr": fpr_at_threshold(family_scores, family_labels, threshold),
        "n_machine": int((family_labels == 1).sum()),
    })

results = pd.DataFrame(rows).round(4)
results

,detector,family,condition,tpr,auroc,test_fpr,n_machine
0,binoculars,BigScience,Zero-shot,0.0178,0.4837,0.0107,2692
1,binoculars,EleutherAI,Zero-shot,0.0007,0.1468,0.0107,1442
2,binoculars,FLAN-T5,Zero-shot,0.0163,0.5268,0.0107,4658
3,binoculars,GLM-130B,Zero-shot,0.3090,0.9458,0.0107,919
4,binoculars,LLaMA,Zero-shot,0.1291,0.9453,0.0107,3710
5,binoculars,OPT,Zero-shot,0.0004,0.3273,0.0107,8012
6,binoculars,OpenAI GPT,Zero-shot,0.2918,0.9266,0.0107,2790


## 6. Summary and save

The saved file uses the same columns as `summary_lexical.csv`, so the two can be
stacked when Chapter 4's tables are built.

The standard deviation columns are left empty rather than set to zero. Zero would
say the seeds were run and made no difference; empty says there were no seeds to
run, which is the accurate statement for a zero-shot detector.

There are seven rows rather than fourteen, because there is only one condition.
When the comparison table is built, each Binoculars figure is repeated across the
in-distribution and LOFO columns, with the caption noting the two are identical by
construction.

In [6]:
print(f"mean TPR across families  {results['tpr'].mean():.3f}")
print(f"worst-case TPR            {results['tpr'].min():.3f} "
      f"({results.loc[results['tpr'].idxmin(), 'family']})")

summary = pd.DataFrame({
    "detector": results["detector"],
    "family": results["family"],
    "condition": results["condition"],
    "tpr_mean": results["tpr"],
    "tpr_std": np.nan,
    "auroc_mean": results["auroc"],
    "auroc_std": np.nan,
    "test_fpr_mean": results["test_fpr"],
    "test_fpr_std": np.nan,
})

out_path = DATA_DIR / "results" / "summary_binoculars.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_path, index=False)

print(f"\nsaved to {out_path}")
summary

mean TPR across families  0.109
worst-case TPR            0.000 (OPT)

saved to data/results/summary_binoculars.csv


,detector,family,condition,tpr_mean,tpr_std,auroc_mean,auroc_std,test_fpr_mean,test_fpr_std
0,binoculars,BigScience,Zero-shot,0.0178,NaN,0.4837,NaN,0.0107,NaN
1,binoculars,EleutherAI,Zero-shot,0.0007,NaN,0.1468,NaN,0.0107,NaN
2,binoculars,FLAN-T5,Zero-shot,0.0163,NaN,0.5268,NaN,0.0107,NaN
3,binoculars,GLM-130B,Zero-shot,0.3090,NaN,0.9458,NaN,0.0107,NaN
4,binoculars,LLaMA,Zero-shot,0.1291,NaN,0.9453,NaN,0.0107,NaN
5,binoculars,OPT,Zero-shot,0.0004,NaN,0.3273,NaN,0.0107,NaN
6,binoculars,OpenAI GPT,Zero-shot,0.2918,NaN,0.9266,NaN,0.0107,NaN


In [7]:
# Is the score tracking length rather than authorship?
test["words"] = test["text"].str.split().str.len()

print("correlation between score and length:",
      round(test["score"].corr(test["words"]), 3))

human_median = test[test["label"] == 0]["words"].median()
print(f"\nhuman test texts, median words: {human_median:.0f}\n")

for family in sorted(test[test["label"] == 1]["family"].dropna().unique()):
    machine = test[test["family"] == family]
    print(f"{family:12s} median words {machine['words'].median():6.0f}  "
          f"ratio to human {machine['words'].median() / human_median:5.2f}  "
          f"mean score {machine['score'].mean():7.4f}")

correlation between score and length: 0.027

human test texts, median words: 116

BigScience   median words     50  ratio to human  0.43  mean score -0.9711
EleutherAI   median words    164  ratio to human  1.41  mean score -1.0896
FLAN-T5      median words     41  ratio to human  0.35  mean score -0.9676
GLM-130B     median words    595  ratio to human  5.13  mean score -0.7728
LLaMA        median words    417  ratio to human  3.59  mean score -0.7917
OPT          median words    145  ratio to human  1.25  mean score -1.0226
OpenAI GPT   median words    109  ratio to human  0.94  mean score -0.7854


In [8]:
# The EleutherAI machine texts Binoculars is most confident are human
worst = test[(test["family"] == "EleutherAI") & (test["label"] == 1)].nsmallest(3, "score")
for _, row in worst.iterrows():
    print(f"--- score {row['score']:.4f}, {row['words']} words, {row['domain']}")
    print(row["text"][:400], "\n")

--- score -1.4061, 121 words, xsum
The club's latest accounts, for the year ending May 2014, also show the Bluebirds owe around PS174m - most of it to owner Vincent Tan. This was despite revenue increasing slightly from 89M in 2013-14 season and net debt dropping by just over 50 (24) during that time period too (to a new low)."What we can say is this has been part one," said Chandler soon after taking his place at Cardiff City Stad 

--- score -1.3978, 64 words, sci_gen
This paper investigates a general framework to discover categories of unlabeled scene images according to their appearances (i.e., textures and structures). We jointly solve the two coupled tasks in an end-toend manner: category discovery from appearance information, as well visual object recognition within each discovered class using conventional classification methods such that accuracy on both  

--- score -1.3762, 110 words, hswag
show the kind of fish they have found as they lay it out on the frozen ground. The ice